In [1]:
import os
import numpy as np
import pandas as pd
import xrayutilities as xu
from xrayutilities.io.imagereader import get_tiff
from xrayutilities.gridder3d import FuzzyGridder3D

# Optional: for VTK export
import vtk
from vtk.util import numpy_support


In [ ]:

# -----------------------------------------------------------------------------
# 0. User switches: SPEC vs Manual metadata
# -----------------------------------------------------------------------------
use_spec = False             # ← set to True to parse a SPEC file
spec_file   = 'my_scan.spec' # your SPEC file
scan_number = 10             # which scan in that SPEC

# If not using SPEC, put your metadata in one of two ways:
#  a) In a CSV file with columns: filename, omega, chi, phi, twotheta
#     meta = pd.read_csv('metadata.csv')
#  b) Directly in Python:
#     meta = pd.DataFrame([
#         {'filename':'img001.tiff','omega':0.0,'chi':0.0,'phi':0.0,'twotheta':30.0},
#         {'filename':'img002.tiff','omega':0.1,'chi':0.0,'phi':0.0,'twotheta':30.1},
#         # ...
#     ])
if not use_spec:
    # Example: manual entry
    meta = pd.DataFrame([
        {'filename':'img001.tiff','omega':0.0,'chi':0.0,'phi':0.0,'twotheta':30.0},
        {'filename':'img002.tiff','omega':0.1,'chi':0.0,'phi':0.0,'twotheta':30.1},
        # add as many frames as you have...
    ])

# -----------------------------------------------------------------------------
# 1. Define crystal & diffractometer geometry
# -----------------------------------------------------------------------------
Si   = xu.materials.Si
Q1   = Si.Q(1, 0, 0)     # in-plane axis
Q2   = Si.Q(0, 0, 1)     # out-of-plane axis
hxrd = xu.HXRD(Q1, Q2)   # builds a 4-circle QConversion
qconv = hxrd._qconv     # use for area detector → (h,k,l)



In [ ]:
# -----------------------------------------------------------------------------
# 2. Prepare your gridder
# -----------------------------------------------------------------------------
h_bins, k_bins, l_bins = 200, 200, 200
gridder = FuzzyGridder3D(h_bins, k_bins, l_bins)  # finite-volume binning

# -----------------------------------------------------------------------------
# 3. Loop: load each frame, convert & bin
# -----------------------------------------------------------------------------
if use_spec:
    # parse SPEC -> returns angles + image dir
    from xrayutilities.io import get_average_RSM
    # This helper will both parse SPEC and populate the gridder internally
    # but if you prefer full manual control, you can extract the metadata via
    # xu.io.specfile.SPECReader and then follow the TIFF loop below.
    grid3D = get_average_RSM(
        h_bins, k_bins, l_bins,
        qconv,
        specfile=spec_file,
        specnr=scan_number,
        datadir='.',
        keepdir=0, roi=None, nav=(1,1), filterfunc=None
    )
    h_axis, k_axis, l_axis = grid3D.xaxis, grid3D.yaxis, grid3D.zaxis
    rsm = grid3D.data

else:
    # manual TIFF + metadata loop
    tiff_folder = 'tiff_folder'
    for _, row in meta.iterrows():
        # read TIFF
        img = get_tiff(os.path.join(tiff_folder, row.filename)).data

        # convert to h,k,l,I arrays
        h, k, l, I = qconv.area(
            img,
            row.omega, row.chi, row.phi, row.twotheta,
            roi=None, normalize=False
        )

        # bin into grid
        gridder(h.ravel(), k.ravel(), l.ravel(), I.ravel())

    # extract axes & data
    h_axis, k_axis, l_axis = gridder.xaxis, gridder.yaxis, gridder.zaxis
    rsm = gridder.data

In [ ]:
# -----------------------------------------------------------------------------
# 4. Export to VTK (.vti) for ParaView / Mayavi
# -----------------------------------------------------------------------------
# Build a vtkImageData (uniform grid) and attach 'Intensity'
H, K, L = rsm.shape
img = vtk.vtkImageData()
img.SetDimensions(H, K, L)

# assume uniform spacing
dh = float(h_axis[1] - h_axis[0])
dk = float(k_axis[1] - k_axis[0])
dl = float(l_axis[1] - l_axis[0])
img.SetSpacing(dh, dk, dl)
img.SetOrigin(float(h_axis[0]), float(k_axis[0]), float(l_axis[0]))

# flatten in Fortran order (VTK expects x-fastest)
flat_int = np.ravel(rsm, order='F').astype(np.float32)
vtk_arr = numpy_support.numpy_to_vtk(
    num_array=flat_int,
    deep=True,
    array_type=vtk.VTK_FLOAT
)
vtk_arr.SetName("Intensity")
img.GetPointData().AddArray(vtk_arr)
img.GetPointData().SetActiveScalars("Intensity")

# write .vti
writer = vtk.vtkXMLImageDataWriter()
writer.SetFileName("rsm_map.vti")
writer.SetInputData(img)
if writer.Write():
    print("Wrote VTK volume: rsm_map.vti")
else:
    raise RuntimeError("Failed to write VTK file")